In [1]:
import pandas as pd
from transformers import T5Tokenizer,Trainer,TrainingArguments,T5ForConditionalGeneration

In [2]:
train_data=pd.read_csv("Dataset/samsum-train.csv")
val_data=pd.read_csv("Dataset/samsum-validation.csv")

In [3]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [4]:
train_data.shape

(14732, 3)

In [5]:
val_data.shape

(818, 3)

In [6]:
#Random sampling

train_data=train_data.sample(n=5000,random_state=42).reset_index(drop=True)
test_data=val_data.sample(n=600,random_state=42).reset_index(drop=True)

In [7]:
train_data.shape


train_data = train_data.dropna(subset=["dialogue", "summary"])
val_data = val_data.dropna(subset=["dialogue", "summary"])  #dropping null
print(train_data["dialogue"].isnull().sum())

0


# Data pre-processing

In [8]:
train_data=train_data.reset_index(drop=True)
val_data=val_data.reset_index(drop=True)

In [9]:
import re

def clean_data(text):
    text=re.sub(r"\r\n"," ",text) #lines
    text=re.sub(r"\s+"," ",text) #spaces
    text=re.sub(r"<.*?>"," ",text)#html tags
    text=text.strip().lower()
    return text
    

In [10]:
train_data["dialogue"]=train_data["dialogue"].apply(clean_data)
train_data["summary"]=train_data["summary"].apply(clean_data)

val_data["dialogue"]=val_data["dialogue"].apply(clean_data)
val_data["summary"]=val_data["summary"].apply(clean_data)

In [11]:
print(train_data["dialogue"][1],"\n")
print(train_data["summary"][1],"\n")

pat: so does anyone know when the stream is going to happen? lou: unfortunately, no, but would really like to. kevin: i don't think i'd be interested in this. pat: y? kevin: seeing all the blood and internal organs makes me dizzy. lou: so you're so gentle? pat: c'mon! srsly? kevin: yup. had the same thing since i was a child. lou: maybe it's time to change it? pat: yeah! give it a try! 

pat and lou are waiting for the stream but kevin is not interested as it makes him dizzy. 



# Tokenize

In [12]:
tokenizer=T5Tokenizer.from_pretrained("t5-small")#T5 small model (lightweight)

In [13]:
# raw data => tokenized inputs for fine tuning

def tokenize(data):
    inputs=tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
    targets=tokenizer(data["summary"],padding="max_length",max_length=150,truncation=True)

    inputs["labels"]=targets["input_ids"]#token ids => add to input as labels

    return inputs

In [14]:
train_dataset=train_data.apply(tokenize,axis=1).tolist()#tolist => because it is compatible with hugging face transformer
test_dataset=val_data.apply(tokenize,axis=1).tolist()

In [15]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [16]:
#inputs ids - dialogue => token ids(dialogue tokens)

#1 => EOS (End of Sequence)

# attention mask 1 => valid token
# labels = target  => summary token

In [17]:
print("Tokens for input : ",len(train_dataset[0]["input_ids"]))
print("Tokens for Labels : ",len(train_dataset[0]["labels"]))

Tokens for input :  512
Tokens for Labels :  150


# Working with our Model

In [18]:
#NLP => generation task but conditional (dialogue) thats why ConditionalGeneration

model=T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [19]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")


In [20]:
print("Device : ",device)
model.to(device)

Device :  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [21]:
#Training Arguments

training_args=TrainingArguments(
    output_dir="./results",
    
    num_train_epochs=10,
    weight_decay=0.01,

    per_device_train_batch_size=8,#default
    per_device_eval_batch_size=8,#default
    eval_strategy="epoch",#When do we want to evaluate? => on end of every epoch
    save_strategy="epoch",
    warmup_steps= 500  # 0 => lr default(5e-05)
)

In [22]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [23]:
# Train or model
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.623541,0.364598
2,0.399709,0.347682
3,0.377262,0.342334
4,0.358388,0.338460
5,0.349775,0.337946
6,0.342352,0.335391
7,0.337661,0.335641
8,0.333216,0.334518
9,0.326958,0.334894
10,0.330653,0.334334


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6250, training_loss=0.6117023754882812, metrics={'train_runtime': 6851.2236, 'train_samples_per_second': 7.298, 'train_steps_per_second': 0.912, 'total_flos': 6767090073600000.0, 'train_loss': 0.6117023754882812, 'epoch': 10.0})

In [25]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [26]:
model=T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer=T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

# Test the core logic for summarization

In [27]:
def summarize_dialogue(dialogue):
    dialogue=clean_data(dialogue) # clean

    # tokenize
    inputs=tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    # generate the summary => (token ids)
    model.to(device)
    target = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=8,
        early_stopping=True,
        length_penalty = 1.2
    )
    #Convert token ids to text(summary) => decoding 
    summary=tokenizer.decode(target[0],skip_special_tokens=True) # Skip EOS,SEP
    return summary 

In [28]:
test_dialog="""
Rahul: Hey Neha, did you finish the assignment for the AI class?

Neha: Not yet. I started it yesterday, but I got stuck on the last question about reinforcement learning.

Rahul: Oh, the one about Q-learning?

Neha: Yeah, exactly. I understand the theory, but implementing it is confusing.

Rahul: Same here. I watched a couple of tutorials, and they helped a bit.

Neha: Can you share them with me?

Rahul: Sure, I’ll send you the links. Also, I think we need to use a simple environment to test it first.

Neha: That makes sense. Maybe something like a grid world?

Rahul: Yeah, that would be easier to debug.

Neha: By the way, when is the submission deadline?

Rahul: It’s tomorrow night.

Neha: Oh no, I need to finish it today then.

Rahul: Don’t worry, we still have time. Let’s work on it together this evening.

Neha: Good idea. That will make it faster.
"""

summary_test=summarize_dialogue(test_dialog)

print("Summary \n",summary_test)

Summary 
 neha started the assignment for the ai class yesterday but got stuck on the last question about reinforcement learning. rahul will send neha the links. the submission deadline is tomorrow night.


In [29]:
test_dialog_2="""
Arjun: Hey Meera, are you free this weekend?

Meera: I think so. Why?

Arjun: I was thinking we could go on a short trip somewhere nearby.

Meera: That sounds nice! Do you have any place in mind?

Arjun: Maybe Mount Abu? It’s not too far and the weather should be good.

Meera: I’ve always wanted to go there. How do you plan to travel?

Arjun: We could either take a train or just drive. Driving might be more flexible.

Meera: True, but it might be tiring.

Arjun: We can take breaks on the way. It’ll be fun.

Meera: Okay, I’m in. When do we leave?

Arjun: Early Saturday morning so we can reach by afternoon.

Meera: Sounds good. What about stay?

Arjun: I’ll book a hotel today.

Meera: Great. Let me know the details.

Arjun: Sure, I’ll share everything once it’s booked.

Meera: I’m excited already!

Arjun: Same here, it’s been a long time since we went on a trip.
"""

summary_test_2=summarize_dialogue(test_dialog_2)

print("Summary \n",summary_test_2)

Summary 
 arjun and meera are going on a short trip to mount abu this weekend. they will take a train or drive. they will leave early saturday morning so they can reach by afternoon.
